In [1]:
from genericbank.spark import get_spark
from genericbank import config

spark = get_spark("exploration")
spark

26/09/27 20:44:43 WARN Utils: Your hostname, TomaszMPC resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/27 20:44:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/tomasz/projects/genericbank-analytics-pyspark/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/tomasz/.ivy2/cache
The jars for the packages stored in: /home/tomasz/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-836b950e-5463-4d27-938a-9704fdc40772;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 262ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [4]:
txn_raw = (
    spark.read
    .option("header", True)
    .csv(str(config.LANDING_DIR / "transactions"))
)
txn_raw.printSchema()


root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- counterparty_country: string (nullable = true)



In [5]:
txn_raw.show(5, truncate=False)
txn_raw.count()

+--------------+----------+-------------------+--------+---------+--------+-------+-----------------+--------------------+
|transaction_id|account_id|txn_ts             |amount  |direction|txn_type|channel|merchant_category|counterparty_country|
+--------------+----------+-------------------+--------+---------+--------+-------+-----------------+--------------------+
|T0000000029   |A00057758 |2025-10-01 06:00:30|6844,07 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000044   |A00003915 |2025-10-01 06:00:44|3959.38 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000047   |A00082374 |2025-10-01 06:00:45|6003.36 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000059   |A00047888 |2025-10-01 06:00:59|11517.76|CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000077   |A00077372 |2025-10-01 06:01:13|2878.18 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
+--------------+

4905630

In [6]:
from pyspark.sql import functions as F
(
    txn_raw
    .groupBy("channel")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-------+-------+
|channel|count  |
+-------+-------+
|POS    |2093278|
|WEB    |1393444|
|MOBILE |869484 |
|ATM    |346752 |
|BRANCH |55306  |
| Pos   |16273  |
|pos    |16107  |
|pos    |16079  |
|Pos    |16071  |
|Web    |10815  |
|web    |10803  |
|web    |10762  |
| Web   |10692  |
|mobile |6911   |
| Mobile|6910   |
|Mobile |6742   |
|mobile |6732   |
|Atm    |2748   |
| Atm   |2722   |
|atm    |2682   |
+-------+-------+
only showing top 20 rows



In [ ]:
dupes = (
    txn_raw
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)
dupes.count()

24427

In [11]:
print("empty amount:     ", txn_raw.filter(F.col("amount").isNull()).count())
print("comma decimal:    ", txn_raw.filter(F.col("amount").contains(",")).count())
print("non-ISO timestamp:", txn_raw.filter(~F.col("txn_ts").rlike(r"^\d{4}-")).count())

empty amount:      4902


comma decimal:     49407


non-ISO timestamp: 98544


In [ ]:
from genericbank import bronze

txn_bronze = bronze.add_ingestion_metadata(bronze.read_transactions(spark), run_id = "data exploration notebook")
txn_bronze.printSchema()
txn_bronze.select("transaction_id", "_source_file", "_ingested_at", "_run_id").show(3, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- counterparty_country: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _run_id: string (nullable = false)

+--------------+-------------------------------------------------------------------------------------------------+--------------------------+
|transaction_id|_source_file                                                                                     |_ingested_at              |
+--------------+-------------------------------------------------------------------------------------------------+--------------------------+
|T0000000029   |file:/home/tomasz/projects/generic